# Bagian 1: Penjelasan Konsep

## 1.1 Kenapa Split Itu Wajib

* Tujuan akhir model: generalisasi akurat pada data baru yang belum pernah dilihat
* Split data: cara mensimulasikan data baru

``` python
Dataset lengkap (yang kamu punya sekarang)
            ↓
    ┌───────────────┬───────────────┐
    │  Training set │  Testing set   │
    │ (model belajar)│ (simulasi     │
    │                │  "data baru") │
    └───────────────┴───────────────┘
```

## 1.2 Split Sebelum Preprocessing

Aturan:
> Split data sebelum melakukan apapun kedata - termasuk sccaling, encoding, imputasi missing value, feature selection, bahkan EDA yang menghasilkan keputusan preprocessing.

``` python
❌ SALAH (leakage):
Dataset lengkap
    ↓
Scaling / Imputasi (dihitung dari SELURUH data)
    ↓
Train/Test Split
    ↓
Model dilatih dengan train
    ↓
Model dievaluasi dengan test
    → Skor evaluasi TERLALU OPTIMISTIS, tidak mencerminkan performa asli

```text
✅ BENAR:
Dataset lengkap
    ↓
Train/Test Split          ← DULUAN
    ↓
Scaling / Imputasi (dihitung HANYA dari train)
    ↓
Model dilatih dengan train
    ↓
Model dievaluasi dengan test (pakai statistik dari train)
    → Skor evaluasi mencerminkan performa asli di data baru
```

## 1.3 Anatomi train_test_split

```pyhton

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    train_size=None,
    random_state=42,
    shuffle=True,
    stratify=None
)
```

* Urutan output **selalu**:
  `X_train, X_test, y_train, y_test`
* **Bukan**:
  `X_train, y_train, X_test, y_test`
* Ini merupakan **urutan baku** pada `train_test_split`.
* Jika urutannya tertukar, kode bisa tetap berjalan tanpa error.
* Namun, fitur dapat tertukar dengan label sehingga hasilnya **kacau total**.
* Ini termasuk **bug tersembunyi yang umum terjadi pada pemula**.


## 1.4 Parameter test_size dan train_size

* test_size: proporsi (0.0–1.0) atau jumlah baris (integer) yang dialokasikan untuk data test
* train_size: kebalikannya — kalau tidak diisi, otomatis dihitung sebagai 1 - test_size

```python
test_size=0.2   # 20% data jadi test, 80% jadi train
test_size=0.3   # 30% data jadi test
test_size=50    # tepat 50 baris jadi test (bukan proporsi)
```

| Ukuran Dataset                     | Rasio Umum Train:Test | Alasan                                                                     |
| ---------------------------------- | --------------------- | -------------------------------------------------------------------------- |
| **Kecil (< 1.000 baris)**          | 70:30 atau 80:20      | Perlu cukup banyak data test agar evaluasi stabil                          |
| **Menengah (1.000–100.000 baris)** | 80:20                 | Standar paling umum digunakan                                              |
| **Sangat besar (jutaan baris)**    | 95:5 atau bahkan 98:2 | 2–5% dari jutaan baris tetap merupakan jumlah yang besar dan representatif |


## 1.5 random_state: Reproducibility

* train_test_split secara default mengacak data sebelum membaginya (shuffle=True)
* Jika tidak menerepkan random_state, setiap kali kode dijalankan ulang:
    * Pembagian data akan berbeda.
    * Baris yang masuk ke train/test akan berbeda
* Akbiatnya hasil ekperimen berubah pada setiap run

``` Pyhton
# Tanpa random_state → hasil split berbeda setiap dijalankan
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Dengan random_state → hasil split SELALU SAMA setiap dijalankan
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

```

Kenapa ini penting secara praktis?

* Reproducibility — kamu (atau orang lain) harus bisa menjalankan ulang kode dan dapat hasil yang identik, untuk debugging maupun untuk kredibilitas hasil eksperimen
* Perbandingan yang adil — kalau kamu membandingkan 3 model berbeda, mereka harus dilatih & diuji dengan pembagian data yang sama persis, kalau tidak, perbedaan skor bisa jadi cuma karena kebetulan pembagian data, bukan karena model yang lebih baik (ini akan sangat relevan di materi 15_model_comparison_experiment)
* Kolaborasi tim — rekan kerja yang menjalankan kode kamu harus dapat hasil yang sama

## 1.6 Parameter shuffle

* Secara default, `train_test_split` menggunakan **`shuffle=True`**.
* Artinya, data **diacak terlebih dahulu** sebelum dibagi menjadi train dan test.
* Pengacakan penting karena data mentah sering tersusun berdasarkan urutan tertentu, misalnya:

  * Urutan tanggal
  * Urutan kelas/target
* Tanpa pengacakan, data test bisa saja hanya berisi **satu jenis kelas**, sehingga evaluasi menjadi tidak representatif.

### Kapan Menggunakan `shuffle=False`?

* Gunakan **`shuffle=False` khususnya pada data time series**.
* Pada time series, **urutan waktu memiliki makna** dan tidak boleh diacak.
* Tujuannya biasanya memprediksi **masa depan berdasarkan masa lalu**.
* Split dilakukan secara **kronologis**:

  * Data lama → **training**
  * Data lebih baru → **testing**
* Contoh:

```text
Waktu ──────────────────────────────→

Data lama              Data baru
───────────────┬────────────────────
    TRAIN      │       TEST
```

* Untuk evaluasi time series, pendekatan yang lebih khusus seperti **Time Series Split** akan dibahas pada materi evaluasi model (`3.5.3`).


## 1.7 stratify: Menjaga Proporsi Kelas

* `stratify` adalah parameter yang **sering dilupakan pemula**, tetapi penting terutama pada **klasifikasi dengan kelas tidak seimbang (imbalanced)**.
* Misalnya dataset memiliki target `churn`:

  * **90%** pelanggan tidak churn → kelas `0`
  * **10%** pelanggan churn → kelas `1`
* Jika melakukan split **tanpa `stratify`**:

  * Proporsi kelas pada train/test dapat berbeda dari proporsi dataset asli.
  * Misalnya data test hanya memiliki **3% churn** atau malah **20% churn**.
* Akibatnya, data test menjadi kurang representatif terhadap distribusi kelas sebenarnya.
* Dengan `stratify=y`, proporsi setiap kelas akan **dipertahankan secara proporsional** pada train dan test.

Contoh:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

Dengan distribusi awal **90:10**, hasil split akan mempertahankan proporsi tersebut secara kira-kira:

```text
Dataset       → 90% kelas 0 | 10% kelas 1
                    ↓
Train         → 90% kelas 0 | 10% kelas 1
Test          → 90% kelas 0 | 10% kelas 1
```

**Intinya:** `stratify=y` membantu memastikan train dan test tetap memiliki **representasi kelas yang sesuai dengan dataset asli**.


Solusi:
```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

Dengan stratify=y, scikit-learn akan memastikan proporsi kelas di train dan test sama persis dengan proporsi di dataset asli.

```text
Dataset asli:     90% kelas 0, 10% kelas 1
Tanpa stratify:   Train 91% / 9%,  Test 87% / 13%   ← proporsi melenceng
Dengan stratify:  Train 90% / 10%, Test 90% / 10%   ← proporsi konsisten
```

Kapan pakai stratify?

* Selalu pertimbangkan untuk kasus klasifikasi, terutama kalau kelasnya tidak seimbang
* Tidak relevan untuk kasus regresi (karena target berupa angka kontinu, bukan kategori) — walau ada teknik stratifikasi versi regresi yang lebih advanced, itu di luar cakupan materi dasar ini

## 1.8 Training vs Validation vs Testing

```text
Dataset lengkap
      ↓
┌───────────┬────────────┬───────────┐
│  Training │ Validation │  Testing  │
│  (± 60%)  │  (± 20%)   │  (± 20%)  │
└───────────┴────────────┴───────────┘
     ↓             ↓            ↓
  Model belajar  Tuning       Evaluasi
  pola dasar    hyperparameter  FINAL
                & pilih model   (sekali saja,
                                di akhir)
```

| Set            | Fungsi                                                             | Berapa Kali Dipakai                 |
| -------------- | ------------------------------------------------------------------ | ----------------------------------- |
| **Training**   | Tempat model belajar parameter (koefisien, dll.)                   | Berkali-kali, tiap eksperimen       |
| **Validation** | Tempat membandingkan model/hyperparameter dan memilih yang terbaik | Berkali-kali, tiap eksperimen       |
| **Testing**    | Mengukur performa **final** setelah semua keputusan model selesai  | **Idealnya hanya sekali**, di akhir |


* **Test set harus tetap independen** dari proses pemilihan dan pengembangan model.
* Jika skor **test set dilihat berulang kali**, informasi tersebut secara tidak langsung memengaruhi keputusan pemilihan model atau hyperparameter.
* Akibatnya, test set tidak lagi benar-benar merepresentasikan **data yang belum pernah digunakan dalam pengambilan keputusan**.
* Kondisi ini disebut **data snooping** dan merupakan bentuk **data leakage yang lebih halus**.
* Karena itu:

  * **Training set** → model belajar.
  * **Validation set** → model dibandingkan dan dipilih.
  * **Test set** → evaluasi final setelah semua keputusan selesai.
* Prinsip utamanya: **jangan menggunakan test set untuk membuat keputusan model**.


## 1.9 Ringkasan Alur Kerja yang Benar

```text
Raw Data
   │
   ▼
Train / Test Split
   │
   ├──────────────────────────────┐
   ▼                              ▼
Training Data                 Test Data
   │                              │
   ▼                              ▼
Preprocessing                 Preprocessing
   │                              │
   │ fit + transform              │ transform saja
   │                              │
   ▼                              │
Model Training                    │
   │                              │
   │ .fit()                       │
   ▼                              ▼
Trained Model ────────────────► Prediction
                                  │
                                  │ .predict()
                                  ▼
                            Evaluasi Test Set

```                           

# Bagian 2: Implementasi

## 2.1 Setup Dataset

In [1]:
from sklearn.datasets import load_iris

import pandas as pd
import numpy as np

In [3]:
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="species")

print("total baris: ", len(X))
print("Distribusi kelas asli: ", y.value_counts())

total baris:  150
Distribusi kelas asli:  species
0    50
1    50
2    50
Name: count, dtype: int64


## 2.2 Split Data

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Ukuran X_train: ", X_train.shape)
print("Ukuran X_test: ", X_test.shape)
print("Ukuran y_train:", y_train.shape)
print("Ukuran y_test :", y_test.shape)

Ukuran X_train:  (120, 4)
Ukuran X_test:  (30, 4)
Ukuran y_train: (120,)
Ukuran y_test : (30,)


## 2.3 Membuktikan Efek random_state

In [8]:
# Split PERTAMA tanpa random_state
split_a = train_test_split(X, y, test_size=0.2)
X_train_a = split_a[0]

# Split KEDUA tanpa random_state
split_b = train_test_split(X, y, test_size=0.2)
X_train_b = split_b[0]

# Bandingkan index baris yang masuk train — kemungkinan besar BEDA
print("Index train split A (5 pertama):", X_train_a.index[:5].tolist())
print("Index train split B (5 pertama):", X_train_b.index[:5].tolist())
print("Apakah identik?", X_train_a.index.tolist() == X_train_b.index.tolist())

Index train split A (5 pertama): [79, 35, 46, 20, 122]
Index train split B (5 pertama): [71, 64, 23, 66, 127]
Apakah identik? False


In [12]:
# Sekarang dengan random_state — jalankan DUA KALI, hasilnya harus SAMA
split_c = train_test_split(X, y, test_size=0.2, random_state=42)
split_d = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_c = split_c[0]
X_train_d = split_d[0]

print(X_train_c.index[:5].tolist())
print(X_train_d.index[:5].tolist())
print("Apakah identik dengan random_state sama?",
      split_c[0].index.tolist() == split_d[0].index.tolist())

[22, 15, 65, 11, 42]
[22, 15, 65, 11, 42]
Apakah identik dengan random_state sama? True


## 2.4 Pentingnya stratify pada Data Tidak Seimbang

In [17]:
# Buat target buatan tidak seimbang: 90% kelas 0, 10% kelas 1

np.random.seed(0)

y_imbalanced = pd.Series(np.random.choice([0, 1], size=1000, p=[0.9, 0.1]))
X_dummy = pd.DataFrame({"fitur": np.random.randn(1000)})

print("Proporsi asli: \n", y_imbalanced.value_counts(normalize=True))

Proporsi asli: 
 0    0.892
1    0.108
Name: proportion, dtype: float64


In [18]:
# Split tanpa stratify
_, _, y_train_no_strat, y_test_no_strat = train_test_split(
    X_dummy, y_imbalanced, test_size=0.2, random_state=42
)

print("\nTanpa stratify — proporsi test:\n", y_test_no_strat.value_counts(normalize=True))


Tanpa stratify — proporsi test:
 0    0.89
1    0.11
Name: proportion, dtype: float64


In [19]:
# Split DENGAN stratify
_, _, y_train_strat, y_test_strat = train_test_split(
    X_dummy, y_imbalanced, test_size=0.2, random_state=42, stratify=y_imbalanced
)
print("\nDengan stratify — proporsi test:\n", y_test_strat.value_counts(normalize=True))


Dengan stratify — proporsi test:
 0    0.89
1    0.11
Name: proportion, dtype: float64


## 2.5 Mengapa Split Harus Duluan? Simulasi Leakage

In [21]:
from sklearn.preprocessing import StandardScaler

#  ❌ CARA SALAH: scaling SEBELUM split
scaler_wrong = StandardScaler()

X_scaled_all = scaler_wrong.fit_transform(X)

X_train_wrong, X_test_wrong, y_train_w, y_test_w = train_test_split(
    X_scaled_all, y, test_size=0.2, random_state=42
)
print("Mean scaler (dihitung dari SELURUH data, termasuk test):", scaler_wrong.mean_)


Mean scaler (dihitung dari SELURUH data, termasuk test): [5.84333333 3.05733333 3.758      1.19933333]


In [22]:
# ✅ CARA BENAR: split DULU, baru scaling
X_train_ok, X_test_ok, y_train_ok, y_test_ok = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler_right = StandardScaler()
X_train_scaled_ok = scaler_right.fit_transform(X_train_ok)   # fit HANYA dari train
X_test_scaled_ok = scaler_right.transform(X_test_ok)
print("Mean scaler (dihitung HANYA dari train):", scaler_right.mean_)

Mean scaler (dihitung HANYA dari train): [5.80916667 3.06166667 3.72666667 1.18333333]


* Pada dataset kecil seperti Iris, perbedaan nilai `mean_` mungkin terlihat kecil.
* Namun, prinsipnya tetap sama: **preprocessing yang belajar dari seluruh dataset telah menerima informasi dari data test**.
* Semakin besar dan kompleks dataset, semakin besar pula potensi distorsi yang ditimbulkan.
* Dampaknya bisa berupa **estimasi performa yang terlalu optimistis** dan tidak mencerminkan kemampuan model pada data baru.
* Karena itu, **split harus dilakukan sebelum preprocessing yang memerlukan `.fit()`**.
* Praktik yang benar perlu dibiasakan sejak awal, bukan menunggu muncul masalah pada proyek yang lebih besar.


## Kesalahan Umum Pemula

| Kesalahan                                                    | Dampaknya                                                                                                |
| ------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------- |
| **Preprocessing** (scaling/imputasi) dilakukan sebelum split | Data leakage → skor evaluasi menjadi tidak jujur                                                         |
| Tidak menetapkan `random_state`                              | Hasil eksperimen tidak dapat direplikasi dan sulit dibandingkan antar model                              |
| Lupa `stratify` pada klasifikasi tidak seimbang              | Proporsi kelas pada test dapat melenceng sehingga evaluasi menjadi bias                                  |
| Tertukar urutan output (`X_train, y_train, X_test, y_test`)  | Fitur dan label tertukar sehingga model belajar hal yang salah tanpa error                               |
| Menggunakan test set berulang kali untuk memilih model       | Test set tidak lagi murni (**data snooping**) → gunakan validation set/cross-validation untuk eksperimen |
| Menggunakan `shuffle=True` pada data time series             | Urutan waktu rusak sehingga evaluasi tidak merepresentasikan skenario forecasting                        |


# Bagian 3: Latihan Praktik

```text
Latihan 1 — Efek test_size
Coba split dataset iris dengan test_size=0.1, 0.3, dan 0.5. Cetak ukuran train/test untuk masing-masing. Menurutmu, test_size berapa yang paling masuk akal untuk dataset sekecil iris (150 baris)? Tulis alasannya.

Latihan 2 — Buktikan random_state Sendiri
Ulangi eksperimen di bagian 2.3, tapi kali ini pakai random_state=0 dan random_state=42 secara terpisah (bukan tanpa random_state). Bandingkan hasil index-nya — apakah beda antar angka random_state yang berbeda? Apakah tetap sama kalau angka random_state-nya sama, dijalankan berkali-kali?

Latihan 3 — Terapkan stratify di Dataset Iris
Dataset iris sebenarnya sudah seimbang (masing-masing kelas 50 baris). Buktikan ini dengan mencetak y_train.value_counts() dan y_test.value_counts(), dengan dan tanpa stratify=y. Apakah hasilnya terlihat beda jauh dibanding eksperimen data tidak seimbang di bagian 2.4? Jelaskan kenapa.

Latihan 4 — Simulasikan Leakage dengan Angka Nyata
Gunakan dataset buatan berikut:

python
np.random.seed(1)
X_leak = pd.DataFrame({"nilai": np.concatenate([np.random.normal(0, 1, 900), np.random.normal(50, 1, 100)])})

Hitung mean() dari seluruh X_leak, lalu bandingkan dengan mean() hanya dari 80% data pertama (setelah displit). Apakah nilainya identik atau berbeda? Jelaskan hubungan hasil ini dengan konsep leakage di bagian 2.5.

Latihan 5 — Rancang Skema Split untuk Kasus Nyata
Bayangkan kamu punya dataset transaksi e-commerce dengan 500.000 baris, target fraud (1% dari transaksi adalah fraud, 99% bukan). Tuliskan (boleh dalam bentuk poin):

test_size berapa yang akan kamu pilih, dan alasannya
Apakah kamu akan pakai stratify? Kenapa?
Apakah kamu perlu validation set terpisah juga? Kenapa